Harry Ferguson - CIS 361 - Summer 2026 - Assignment 7

In [1]:
import warnings
warnings.filterwarnings('ignore')
import nltk
import numpy as np
import pandas as pd
from keras.models import Sequential
from keras.layers import LSTM, Embedding, Dense
from keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split

csv = pd.read_csv('ner/ner_dataset.csv', encoding='latin1')
csv.head(25)

,Sentence #,Word,POS,Tag
0,Sentence: 1,Thousands,NNS,O
1,NaN,of,IN,O
2,NaN,demonstrators,NNS,O
3,NaN,have,VBP,O
4,NaN,marched,VBN,O
5,NaN,through,IN,O
6,NaN,London,NNP,B-geo
7,NaN,to,TO,O
8,NaN,protest,VB,O
9,NaN,the,DT,O


In [2]:
X, Y, X_sentence, Y_sentence = [], [], [] ,[]

for row in csv.itertuples():
    if isinstance(row._1, str) and row._1 != 'Sentence: 1':
        X.append(X_sentence)
        Y.append(Y_sentence)
        X_sentence, Y_sentence = [], []
    X_sentence.append(str(row.Word).lower())
    Y_sentence.append(row.Tag)
print(X[0])
print(Y[0])

['thousands', 'of', 'demonstrators', 'have', 'marched', 'through', 'london', 'to', 'protest', 'the', 'war', 'in', 'iraq', 'and', 'demand', 'the', 'withdrawal', 'of', 'british', 'troops', 'from', 'that', 'country', '.']
['O', 'O', 'O', 'O', 'O', 'O', 'B-geo', 'O', 'O', 'O', 'O', 'O', 'B-geo', 'O', 'O', 'O', 'O', 'O', 'B-gpe', 'O', 'O', 'O', 'O', 'O']


In [3]:
max_len = max([len(s) for s in X])
print('Maximum sentence length: ', max_len)

tags = set(label for sentence_labels in Y for label in sentence_labels)
tag2idx = {tag: i for i, tag in enumerate(tags)}
idx2tag = {i: tag for tag, i in tag2idx.items()}

words = set(word for sentence in X for word in sentence)
word2idx = {word: i + 1 for i, word in enumerate(words)}
word2idx['<PAD>'] = 0
idx2word = {i: word for word, i in word2idx.items()}

x = [[word2idx[word] for word in sentence] for sentence in X]
y = [[tag2idx[tag] for tag in tags] for tags in Y]

print('X before padding: ',x[0])
print('Y before padding: ',y[0])

x = pad_sequences(x, maxlen=max_len, padding='post')
y = pad_sequences(y, maxlen=max_len, padding='post')

print('X after padding: ',x[0])
print('Y after padding: ',y[0])


Maximum sentence length:  104
X before padding:  [19786, 2039, 22183, 26387, 31330, 11894, 26679, 1737, 1081, 31261, 29862, 15017, 26983, 17009, 1653, 31261, 2684, 2039, 16982, 5433, 23989, 22296, 8705, 21248]
Y before padding:  [11, 11, 11, 11, 11, 11, 13, 11, 11, 11, 11, 11, 13, 11, 11, 11, 11, 11, 3, 11, 11, 11, 11, 11]
X after padding:  [19786  2039 22183 26387 31330 11894 26679  1737  1081 31261 29862 15017
 26983 17009  1653 31261  2684  2039 16982  5433 23989 22296  8705 21248
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0     0     0     0     0     0     0     0     0     0     0
     0     0     0     0     0     0     0     0]
Y after padding:  [11 1

In [4]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2)

# The LSTM model
model = Sequential()
model.add(Embedding(input_dim=len(x)+1, output_dim=50, input_length=max_len))
model.add(LSTM(units=100, return_sequences=True))
model.add(Dense(len(y), activation='softmax'))
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.fit(x_train, y_train, validation_data=(x_test, y_test), epochs=3, batch_size=32)
loss, accuracy = model.evaluate(x_test, y_test)
print(f'Loss: {loss}, Accuracy: {accuracy}')

Epoch 1/3
1199/1199 ━━━━━━━━━━━━━━━━━━━━ 2553s 2s/step - accuracy: 0.9156 - loss: 0.6150 - val_accuracy: 0.9680 - val_loss: 0.1700
Epoch 2/3
1199/1199 ━━━━━━━━━━━━━━━━━━━━ 2605s 2s/step - accuracy: 0.9697 - loss: 0.1306 - val_accuracy: 0.9754 - val_loss: 0.0923
Epoch 3/3
1199/1199 ━━━━━━━━━━━━━━━━━━━━ 2690s 2s/step - accuracy: 0.9797 - loss: 0.0733 - val_accuracy: 0.9828 - val_loss: 0.0658
300/300 ━━━━━━━━━━━━━━━━━━━━ 328s 1s/step - accuracy: 0.9828 - loss: 0.0658
Loss: 0.0657770037651062, Accuracy: 0.9828333854675293
